# 示例: 2D水动力模型(概念验证)

**脚本:** `examples/run_2d_model.py`

## 目的
此示例用于演示项目中的2D水动力模型（`model_2d/`）的基本用法。这是一个概念验证（Proof-of-Concept）级别的模型，用于在一个非结构化的三角网格上求解浅水方程。

此笔记本将展示如何：
1.  从点和三角形的定义来手动构建一个简单的计算网格（`Mesh`）。
2.  为一个经典的“溃坝”问题设置初始条件。
3.  实例化并运行2D模型。
4.  检查并可视化最终的计算结果（水深）。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from model_2d.mesh import Mesh
from model_2d.model import Model2D

## 2. 构建计算网格

2D模型在非结构化的三角网格上进行计算。我们首先定义网格的节点（points）和组成网格的三角形（triangles），然后用这些信息来构建一个`Mesh`对象。`Mesh`对象会自动计算出网格的拓扑关系，如边、单元等。

In [ ]:
points = [
    (0,0), (1,0), (2,0), (3,0), # Row 1
    (0,1), (1,1), (2,1), (3,1), # Row 2
    (0,2), (1,2), (2,2), (3,2)  # Row 3
]

triangles = [
    (0,1,4), (1,5,4), # Square 1
    (1,2,5), (2,6,5), # Square 2
    (2,3,6), (3,7,6), # Square 3
    (4,5,8), (5,9,8), # Square 4
    (5,6,9), (6,10,9),# Square 5
    (6,7,10),(7,11,10)# Square 6
]

mesh = Mesh()
mesh.build_from_points_and_triangles(points, triangles)

print(f"网格构建完成，包含 {len(mesh.faces)} 个计算单元。")

## 3. 设置初始条件

我们来模拟一个简单的“溃坝”问题。我们将计算区域左半部分的水深(`h`)设为2.0米，右半部分设为1.0米，模拟一个高水位的区域和一个低水位的区域。初始流速(`uh`, `vh`)和河床高程(`z_bed`)都为0。

In [ ]:
print("设置溃坝问题的初始条件...")
for face in mesh.faces:
    if face.centroid[0] < 1.5:
        face.h = 2.0
    else:
        face.h = 1.0

initial_depths = np.array([f.h for f in mesh.faces])

## 4. 实例化并运行模型

我们创建一个`Model2D`的实例，然后在一个循环中调用其`step`方法来推进模拟。2D模型为了保证计算稳定，通常需要较小的时间步长(`dt`)。

In [ ]:
model = Model2D(name="2D_dam_break", mesh=mesh)

num_steps = 20
dt = 0.05

print(f"运行模拟 {num_steps} 步...")
for i in range(num_steps):
    model.step(inflows={}, dt=dt)
    if (i+1) % 5 == 0:
        h_left = model.mesh.faces[0].h
        h_right = model.mesh.faces[-1].h
        print(f"Step {i+1}: 左侧单元水深 = {h_left:.3f}, 右侧单元水深 = {h_right:.3f}")

print("\n模拟完成。")

## 5. 检查并可视化结果

模拟结束后，我们可以检查每个计算单元的最终水深。为了更直观地展示结果，我们绘制了计算网格，并用颜色深浅来表示每个单元的最终水深。可以看到，水从左侧流向右侧，左边的水深降低，右边的水深升高，符合物理预期。

In [ ]:
final_depths = np.array([f.h for f in model.mesh.faces])

print("每个单元的最终水深 (h):")
for face in model.mesh.faces:
    print(f"  单元 {face.id:2d} @ ({face.centroid[0]:.2f}, {face.centroid[1]:.2f}): h = {face.h:.4f}")

# 可视化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

points_arr = np.array(points)

# 绘制初始状态
ax1.tripcolor(points_arr[:, 0], points_arr[:, 1], triangles, facecolors=initial_depths, cmap='viridis', edgecolors='k')
ax1.set_title('Initial Water Depth')
ax1.set_xlabel('X coordinate')
ax1.set_ylabel('Y coordinate')
ax1.set_aspect('equal', 'box')

# 绘制最终状态
tpc = ax2.tripcolor(points_arr[:, 0], points_arr[:, 1], triangles, facecolors=final_depths, cmap='viridis', edgecolors='k')
ax2.set_title('Final Water Depth')
ax2.set_xlabel('X coordinate')
ax2.set_aspect('equal', 'box')

fig.colorbar(tpc, ax=ax2, label='Water Depth (h)')
plt.tight_layout()
plt.show()